# Solución 7: Integración — Cohete analítico y Sumas de Riemann

Origen: `01. Fundamental Algorithms/03. Integration/Integracion.pdf`

---

## Parte 1: Integración analítica — distancia recorrida por un cohete

Un cohete asciende verticalmente y expulsa combustible a una tasa de $2100\,\text{kg/s}$.
La masa inicial es $m_0 = 140\,000\,\text{kg}$.
Partiendo del reposo en $t=0$, la velocidad ascendente está dada por:

$$v(t) = 2000\ln\!\left(\frac{140000}{140000-2100t}\right) - 9.8t \quad [\text{m/s}]$$

La distancia recorrida desde $t_1=8\,\text{s}$ hasta $t_2=30\,\text{s}$ es:

$$x = \int_{8}^{30} \left[2000\ln\!\left(\frac{140000}{140000-2100t}\right) - 9.8t\right] dt$$

### Solución analítica

Usamos la identidad $\int \ln(u)\,du = u\ln(u) - u + C$.

Sea $u = 140000 - 2100t$, entonces $du = -2100\,dt$ y:

$$\int 2000\ln\!\left(\frac{140000}{u}\right)\frac{du}{-2100}$$

Desarrollando y evaluando en los límites se obtiene el valor exacto.

In [ ]:
import numpy as np
from scipy import integrate

# ============================================================
# PARÁMETROS DEL COHETE
# ============================================================
m0   = 140_000   # masa inicial [kg]
q    = 2_100     # tasa de expulsión de combustible [kg/s]
ve   = 2_000     # velocidad de expulsión [m/s]
g    = 9.8       # gravedad [m/s²]
t1, t2 = 8, 30  # intervalo de integración [s]

def v(t):
    """Velocidad ascendente del cohete [m/s]."""
    return ve * np.log(m0 / (m0 - q*t)) - g*t

# ============================================================
# INTEGRACIÓN ANALÍTICA EXACTA CON SCIPY
# (referencia / valor exacto)
# ============================================================
x_exacta, error = integrate.quad(v, t1, t2)

print("=" * 55)
print("INTEGRACIÓN ANALÍTICA — distancia recorrida por el cohete")
print("=" * 55)
print(f"  Intervalo           : t = [{t1}, {t2}] s")
print(f"  Distancia exacta    : {x_exacta:.4f} m")
print(f"  Error estimado scipy: {error:.2e} m")
print("=" * 55)

In [ ]:
import matplotlib.pyplot as plt

t_vals = np.linspace(t1, t2, 300)
v_vals = v(t_vals)

plt.figure(figsize=(9, 5))
plt.fill_between(t_vals, v_vals, alpha=0.25, color='royalblue', label='Área = distancia')
plt.plot(t_vals, v_vals, color='royalblue', linewidth=2.5, label=r'$v(t)$')
plt.axvline(t1, color='gray', linestyle='--', linewidth=1)
plt.axvline(t2, color='gray', linestyle='--', linewidth=1)
plt.title('Velocidad del cohete y área integrada', fontsize=13)
plt.xlabel('Tiempo [s]')
plt.ylabel('Velocidad [m/s]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Distancia recorrida (integración analítica): {x_exacta:.2f} m")

---

## Parte 2: Sumas de Riemann — $y = x^3$, $x \in [0, b]$

**Ejercicio para entregar:** Halla el área exacta de la región entre la parábola $y = x^3$ y el eje $x$ en el intervalo $[0, b]$.
Usar las **sumas de Riemann a la derecha** con segmentos de igual ancho.

### Solución analítica de referencia

$$\int_0^b x^3\,dx = \frac{b^4}{4}$$

### Sumas de Riemann

Con $n$ subintervalos de ancho $\Delta x = b/n$, los puntos derechos son $x_i = i\,\Delta x$ para $i=1,\ldots,n$:

$$R_n = \sum_{i=1}^{n} f(x_i)\,\Delta x = \sum_{i=1}^{n} (i\,\Delta x)^3\,\Delta x$$

In [ ]:
def riemann_derecha(f, a, b, n):
    """
    Suma de Riemann a la DERECHA para f en [a, b] con n subintervalos.
    Devuelve (aproximación, dx).
    """
    dx = (b - a) / n
    x_derechos = np.array([a + i * dx for i in range(1, n + 1)])
    return np.sum(f(x_derechos) * dx), dx


# ============================================================
# PARÁMETROS
# ============================================================
b = 4.0           # Límite superior (elección propia)
f_cubic = lambda x: x**3

exacto = b**4 / 4
print(f"Valor exacto de ∫₀^{b} x³ dx = b⁴/4 = {exacto:.6f}")
print()

# Convergencia al aumentar n
ns = [2, 5, 10, 50, 100, 500, 1000]
print(f"{'n':>6} | {'Riemann (der.)':>16} | {'Error abs.':>12} | {'Error rel. %':>14}")
print("-" * 58)
for n in ns:
    R, dx = riemann_derecha(f_cubic, 0, b, n)
    err_abs = abs(R - exacto)
    err_rel = err_abs / exacto * 100
    print(f"{n:>6} | {R:>16.6f} | {err_abs:>12.6f} | {err_rel:>13.4f}%")

In [ ]:
# Visualización con n=10 (para ver claramente los rectángulos)
n_vis = 10
R_vis, dx_vis = riemann_derecha(f_cubic, 0, b, n_vis)

x_cont = np.linspace(0, b, 400)

fig, ax = plt.subplots(figsize=(9, 5))

# Rectángulos de Riemann
x_derechos = np.linspace(dx_vis, b, n_vis)
for xr in x_derechos:
    xl = xr - dx_vis
    ax.bar(xl, f_cubic(xr), width=dx_vis, align='edge',
           color='steelblue', alpha=0.4, edgecolor='steelblue', linewidth=0.8)

ax.plot(x_cont, f_cubic(x_cont), 'k-', linewidth=2.5, label=r'$f(x)=x^3$')
ax.fill_between(x_cont, f_cubic(x_cont), alpha=0.1, color='black')
ax.set_title(f'Sumas de Riemann (derecha) para $x^3$ con $n={n_vis}$, $b={b}$', fontsize=13)
ax.set_xlabel('$x$')
ax.set_ylabel('$f(x)$')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nRiemann (n={n_vis}): {R_vis:.6f}")
print(f"Exacto           : {exacto:.6f}")
print(f"Error absoluto   : {abs(R_vis - exacto):.6f}")

In [ ]:
# Gráfica de convergencia: error vs n
ns_conv = np.logspace(0, 4, 60, dtype=int)
errores = [abs(riemann_derecha(f_cubic, 0, b, n)[0] - exacto) for n in ns_conv]

plt.figure(figsize=(8, 5))
plt.loglog(ns_conv, errores, 'o-', color='crimson', linewidth=2, markersize=4, label='Error Riemann')
# Línea de referencia O(1/n)
ref = errores[0] * ns_conv[0] / ns_conv
plt.loglog(ns_conv, ref, 'k--', linewidth=1, label=r'$\mathcal{O}(1/n)$')
plt.xlabel('Número de subintervalos $n$')
plt.ylabel('Error absoluto')
plt.title('Convergencia de la suma de Riemann', fontsize=13)
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()